# Notebook 38 — Resilience: Deadlines & Retry

Keep workflows responsive and bounded in time even when components are flaky
or slow.

| Component | Role |
|---|---|
| `RetryPolicy` | Exponential backoff with jitter for individual calls |
| `RetryResult` | Outcome: attempts, total delay, error |
| `Deadline` | A fixed time budget with helpers (`remaining_s`, `is_expired`) |
| `DeadlineGuard` | Wraps any async function with a hard timeout |
| `DeadlineManager` | Tracks deadlines for many concurrent runs |
| `WorkflowRetry` | Full-workflow retry loop with `RetryPolicy` + `Deadline` |
| `WorkflowRetryResult` | Outcome with `success`, `attempts`, `deadline_breached` |

## Setup

In [ ]:
import sys
sys.path.insert(0, '../sdk')

from multigen.resilience import (
    RetryPolicy,
    RetryResult,
    Deadline,
    DeadlineError,
    DeadlineGuard,
    DeadlineManager,
    WorkflowRetry,
    WorkflowRetryResult,
)

print('All imports OK')

---
## Section 1 — RetryPolicy Basics

`RetryPolicy` implements exponential backoff:

```
delay(attempt) = min(max_delay, base_delay × multiplier^attempt) + jitter
```

Pass any async or sync callable to `policy.call(fn)` — it handles retries
transparently and returns a `RetryResult` with the final output, attempt count,
and total delay incurred.

In [ ]:
# Inspect the backoff schedule for different attempt numbers
policy = RetryPolicy(
    max_retries=5,
    base_delay=1.0,
    multiplier=2.0,
    max_delay=30.0,
    jitter=0.0,   # disable jitter for deterministic preview
)

print('Backoff schedule (jitter=0):')
print(f'  {"Attempt":>8}  {"Delay (s)":>12}')
for attempt in range(6):
    print(f'  {attempt+1:>8}  {policy.delay_for(attempt):>12.2f}')

In [ ]:
import asyncio

# Simulate a flaky function that fails the first 2 times, then succeeds
flaky_call_count = {'n': 0}

async def flaky_llm_call():
    flaky_call_count['n'] += 1
    n = flaky_call_count['n']
    if n <= 2:
        raise ConnectionError(f'LLM API timeout (attempt {n})')
    return {'answer': 'The answer is 42', 'attempt': n}

retry_log = []

policy_with_cb = RetryPolicy(
    max_retries=4,
    base_delay=0.01,  # very short for demo purposes
    multiplier=2.0,
    max_delay=1.0,
    jitter=0.0,
    on_retry=lambda attempt, exc, delay: retry_log.append(
        f'Retry {attempt}: {exc} — waiting {delay:.3f}s'
    ),
)

async def demo_retry():
    result = await policy_with_cb.call(flaky_llm_call)
    return result

result = asyncio.run(demo_retry())

print('Retry log:')
for entry in retry_log:
    print(f'  {entry}')
print()
print(f'Final result: {result.output}')
print(f'Total attempts: {result.attempts}')
print(f'Total delay:    {result.total_delay_s:.3f}s')
print(f'Success:        {result.success}')

---
## Section 2 — Deadline Tracking

`Deadline` creates a monotonic time budget.  It never drifts because it uses
`time.monotonic()` internally.

Key methods:
- `remaining_s()` — seconds left (0.0 once expired)
- `is_expired()` — True / False
- `assert_not_expired(context)` — raises `DeadlineError` if expired
- `fraction_used()` — 0.0 → 1.0 progress indicator

In [ ]:
import time

# Create a 5-second deadline
d = Deadline(budget_s=5.0)
print(f'Created: {d}')
print(f'Remaining:      {d.remaining_s():.3f}s')
print(f'Elapsed:        {d.elapsed_s():.3f}s')
print(f'Fraction used:  {d.fraction_used():.3f}')
print(f'Is expired:     {d.is_expired()}')

# Simulate some work
time.sleep(0.05)
print()
print(f'After 50ms work:')
print(f'Remaining:      {d.remaining_s():.3f}s')
print(f'Elapsed:        {d.elapsed_s():.3f}s')
print(f'Fraction used:  {d.fraction_used():.3f}')

# assert_not_expired should pass (lots of time left)
try:
    d.assert_not_expired('section-2-demo')
    print('assert_not_expired: OK (not expired yet)')
except DeadlineError as e:
    print(f'DeadlineError: {e}')

# Show a deadline that has already expired
expired = Deadline(budget_s=0.001)
time.sleep(0.01)
print()
print(f'Expired deadline: is_expired={expired.is_expired()}, remaining={expired.remaining_s():.3f}s')
try:
    expired.assert_not_expired('pre-flight-check')
except DeadlineError as e:
    print(f'Caught DeadlineError: {e}')

---
## Section 3 — DeadlineGuard

`DeadlineGuard` wraps any async function with a hard timeout equal to the
deadline's remaining budget.  If the function takes longer than the remaining
time it is cancelled and a `DeadlineError` is raised.

An optional `on_breach` callback lets you trigger an alert or escalation.

In [ ]:
async def fast_agent(ctx: dict) -> dict:
    await asyncio.sleep(0.01)  # 10ms — should complete in time
    return {'answer': 'fast result', 'query': ctx.get('query')}

async def slow_agent(ctx: dict) -> dict:
    await asyncio.sleep(10.0)  # 10s — will exceed budget
    return {'answer': 'slow result'}

# Test 1: fast agent within deadline
async def test_fast():
    d = Deadline(budget_s=2.0)
    guard = DeadlineGuard(d, on_breach=lambda dl: print(f'  [ALERT] SLA breached after {dl.elapsed_s():.2f}s'))
    result = await guard.run(fast_agent, {'query': 'hello'})
    return result

result = asyncio.run(test_fast())
print(f'Fast agent result: {result}')

# Test 2: slow agent exceeds deadline
async def test_slow():
    d = Deadline(budget_s=0.05)  # only 50ms budget
    breach_log = []
    guard = DeadlineGuard(
        d,
        on_breach=lambda dl: breach_log.append(
            f'Breached at {dl.elapsed_s():.3f}s (budget={dl.budget_s}s)'
        ),
    )
    try:
        await guard.run(slow_agent, {})
    except DeadlineError as e:
        return str(e), breach_log

err, log = asyncio.run(test_slow())
print(f'Slow agent raised: {err}')
print(f'Breach log: {log}')

---
## Section 4 — DeadlineManager

`DeadlineManager` acts as a registry of deadlines for multiple concurrent
workflow runs, identified by string `run_id`.  It provides convenience methods
to create, inspect, guard, and cancel deadlines.

Typical use: create a deadline per incoming API request, guard each agent call
against it, and cancel on completion.

In [ ]:
async def demo_manager():
    breach_events = []
    mgr = DeadlineManager(default_budget_s=60.0)
    mgr.on_breach(lambda d: breach_events.append('breached'))

    # Create deadlines for three concurrent runs
    d_fast  = mgr.create('run-001', budget_s=2.0)
    d_long  = mgr.create('run-002', budget_s=120.0)
    d_tight = mgr.create('run-003', budget_s=0.001)  # will expire immediately

    print('=== Deadlines created ===')
    for run_id, info in mgr.status().items():
        print(f'  {run_id}: remaining={info["remaining_s"]:.3f}s  expired={info["expired"]}')

    # Guard run-001 with a fast async task
    result = await mgr.guard('run-001').run(fast_agent, {'query': 'test'})
    print(f'\nrun-001 result: {result}')

    # run-003 has already expired
    time.sleep(0.01)
    print(f'\nExpired runs: {mgr.expired_runs()}')

    # Cancel run-002 (e.g. client disconnected)
    mgr.cancel('run-002')
    print(f'After cancel run-002, status keys: {list(mgr.status().keys())}')

print('=== DeadlineManager Demo ===')
asyncio.run(demo_manager())

---
## Section 5 — WorkflowRetry: Full Retry Loop with Deadline

`WorkflowRetry` combines `RetryPolicy` (per-attempt backoff) with a `Deadline`
(overall time budget).  Unlike retrying a single function, this retries an
**entire workflow chain** on failure.

Result fields:
- `success` — True only if no error and no deadline breach
- `attempts` — how many times the workflow was attempted
- `deadline_breached` — True if the budget ran out mid-run
- `total_elapsed_s` — wall-clock time consumed

In [ ]:
# Simulate a workflow that fails twice then succeeds
wf_attempts = {'n': 0}

async def my_workflow(ctx: dict) -> dict:
    wf_attempts['n'] += 1
    n = wf_attempts['n']
    # Fails on first two full attempts
    if n <= 2:
        raise RuntimeError(f'Workflow failed on attempt {n} — upstream service unavailable')
    return {
        'summary': f'Successfully processed: {ctx.get("query", "")}',
        'attempt': n,
    }

retry_events = []

async def run_workflow_retry():
    policy = RetryPolicy(
        max_retries=4,
        base_delay=0.01,   # short for demo
        multiplier=2.0,
        max_delay=0.5,
        jitter=0.0,
    )
    deadline = Deadline(budget_s=10.0)  # generous budget

    wr = WorkflowRetry(
        policy=policy,
        deadline=deadline,
        on_breach=lambda: print('  [ONCALL] Workflow deadline breached!'),
        on_retry=lambda attempt, exc: retry_events.append(
            f'Attempt {attempt} failed: {exc}'
        ),
    )

    result = await wr.run(my_workflow, {'query': 'Summarise Q4 earnings report'})
    return result

result = asyncio.run(run_workflow_retry())

print('=== WorkflowRetryResult ===')
print(f'  success:           {result.success}')
print(f'  attempts:          {result.attempts}')
print(f'  deadline_breached: {result.deadline_breached}')
print(f'  total_elapsed_s:   {result.total_elapsed_s:.3f}s')
print(f'  output:            {result.output}')
print()
print('Retry events:')
for event in retry_events:
    print(f'  {event}')

In [ ]:
# Show deadline-breached case: tight budget that expires before retries finish
wf_always_slow_count = {'n': 0}

async def always_slow_workflow(ctx: dict) -> dict:
    wf_always_slow_count['n'] += 1
    await asyncio.sleep(0.1)  # 100ms per attempt
    raise RuntimeError('Service still down')

async def run_breached_deadline():
    policy = RetryPolicy(max_retries=10, base_delay=0.0, max_delay=0.0, jitter=0.0)
    deadline = Deadline(budget_s=0.15)  # only 150ms — not enough for 2 retries

    wr = WorkflowRetry(policy=policy, deadline=deadline)
    result = await wr.run(always_slow_workflow, {})
    return result

br = asyncio.run(run_breached_deadline())
print('=== Deadline Breached Case ===')
print(f'  success:           {br.success}')
print(f'  deadline_breached: {br.deadline_breached}')
print(f'  attempts made:     {br.attempts}')
print(f'  elapsed:           {br.total_elapsed_s:.3f}s')

---
## Summary

```
RetryPolicy          →  per-call exponential backoff (configurable multiplier + jitter)
Deadline             →  monotonic budget with remaining_s() / is_expired()
DeadlineGuard        →  asyncio.wait_for wrapper with breach callback
DeadlineManager      →  registry for many concurrent run deadlines
WorkflowRetry        →  full-workflow retry + deadline combined
```

**Recommended production wiring:**
```python
policy   = RetryPolicy(max_retries=3, base_delay=1.0, max_delay=30.0)
deadline = Deadline(budget_s=120)
wr = WorkflowRetry(policy=policy, deadline=deadline,
                   on_breach=lambda: alert_pagerduty())

result = await wr.run(my_workflow_fn, ctx)
if result.deadline_breached:
    return error_response('Workflow timed out')
if not result.success:
    return error_response(str(result.error))
return result.output
```